In [ ]:
print("hello world")

# Simple SQL Demo with Shapefiles

This notebook demonstrates working with shapefiles in DuckDB, showing how to extract geometric properties and visualize them on maps.

## Setup

First, let's import the necessary libraries and set up our DuckDB connection with the spatial extension.

In [ ]:
import folium
from IPython.display import display
import json
from duckdb import connect

# Use shared connection with extensions loaded
from duckdb_setup.connection import get_readonly_connection
if 'con' not in locals() or con is None:
    con = get_readonly_connection()
else:
    print("Using existing connection")

### Source Data

In [ ]:
# !python duckdb_setup/5-shapes.py

In [ ]:
# Look into a script that will download the needed files (5-shapes.py)
# Load shapefile into DuckDB
# con.execute("""
#     CREATE OR REPLACE TABLE states AS
#     SELECT * FROM st_read('project_data/shapes/tl_2025_us_state/tl_2025_us_state.shp')
# """)

# Verify the data loaded
state_count = con.execute("SELECT COUNT(*) as count FROM states").fetchall()

print(f"There are {format(state_count[0][0])} states in the shapefile.")

In [ ]:
# con.execute("SELECT * FROM states where STUSPS = 'OH'").fetchdf()
# con.execute("SELECT * FROM states where region <> 9 and stusps <> 'HI' and stusps <> 'AK'").fetchdf()
con.execute("Select * from pragma_database_list").fetchdf()
con.execute("SHOW TABLES").fetchdf()

## Quick Example

In [ ]:
# Create a map and show a shape
quick_map = folium.Map(location=[40.12, -83.07], zoom_start=12)

# Add hardcoded polygon shape
hardcoded_shape = {
    "type": "Polygon",
    "coordinates": [[
        [-83.10, 40.15],
        [-83.05, 40.18],
        [-83.02, 40.12],
        [-83.06, 40.08],
        [-83.09, 40.10],
        [-83.10, 40.15]
    ]]
}

# folium.GeoJson(hardcoded_shape, style_function=lambda x: {'color': 'red', 'fillColor': 'red', 'fillOpacity': 0.3}).add_to(quick_map)
folium.GeoJson(hardcoded_shape).add_to(quick_map)

save_path = 'maps/quick_map.html'
quick_map.save(save_path)
quick_map

## Simple SQL Demo

### Create Table - Sample Shape

In [ ]:
con_simple = connect(database=':memory:')
con_simple.execute(""" LOAD spatial;""");
                   
con_simple.execute("""DROP TABLE IF EXISTS sample_shape; CREATE TABLE sample_shape (
    id INTEGER,
    geom GEOMETRY
);""")

con_simple.execute("""INSERT INTO sample_shape VALUES (
    1,
    ST_GeomFromText('POLYGON((-83.10 40.15, -83.05 40.18, -83.02 40.12, -83.06 40.08, -83.09 40.10, -83.10 40.15))')
);""")


### Query Table - Sample Shape

In [ ]:
sample_shape = con_simple.execute("SELECT id, geom, ST_AsText(geom) as wkt_geometry, ST_AsGeoJSON(geom) as geojson_outline, FROM sample_shape").fetchdf()
sample_shape

### Sample Shape - On a Map

In [ ]:
# Create a map and show a shape
sample_shape_map = folium.Map(location=[40.12, -83.07], zoom_start=12)

# Add sample_shape geojson
sample_shape['geojson_outline'][0]

folium.GeoJson(sample_shape['geojson_outline'][0]).add_to(sample_shape_map)

save_path = 'maps/sample_shape_map.html'
sample_shape_map.save(save_path)
sample_shape_map

## No Map Required (But Still Helps)

### Centroid

What is it: The centroid is the geometric center of a shape.
Please note, the Census Bureau files include INTPTLAT and INTPTLON fields which are latitude and longitude representations of the interior point. The interior point is very similar to the centroid, but may have been adjusted to account for water areas or edge cases where the centroid is not actually inside of the shape (e.g., crescent shapes).

In [ ]:
# Extract the centroid
centroid_query = """
    SELECT 
        NAME,
        ST_AsGeoJSON(geom) as geom_outline,
        ST_Centroid(geom) as centroid_geom,
        st_x(centroid_geom) as centroid_lon,
        st_y(centroid_geom) as centroid_lat
    FROM states
    WHERE NAME = 'Ohio'
"""

centroid_data = con.execute(centroid_query).fetchdf()
display(centroid_data)

In [ ]:
# Create map with shape, centroid, and bounding box
centroid_map = folium.Map(location=[centroid_data['centroid_lat'][0], centroid_data['centroid_lon'][0]], zoom_start=7)

# Add state boundary (filled)
folium.GeoJson(centroid_data['geom_outline'][0], style_function=lambda x: {'fillColor': 'lightblue'}).add_to(centroid_map)

# Add centroid marker
folium.Marker(location=[centroid_data['centroid_lat'][0], centroid_data['centroid_lon'][0]], icon=folium.Icon(color='blue')).add_to(centroid_map)

save_path = 'maps/centroid_map.html'
centroid_map.save(save_path)
centroid_map

### Bounding Box

- A bounding box, or envelope, is derived from a geometry or collection of points.
- Returns a set of points to make a rectangular polygon around the shape.
- Commonly used as an aggregate function to find the bounding box of multiple geometries
- Very useful to prefilter points in relation to a shape before more precise filtering. e.g., verify addresses in a specific neighborhood of Cleveland by pre-filtering all addresses for the neighborhood's bounding box.


In [ ]:
# Extract the bounding box
bbox_query = """
    SELECT 
        NAME,
        ST_AsGeoJSON(geom) as geom_outline,
        ST_Centroid(geom) as centroid_geom,
        st_x(centroid_geom) as centroid_lon,
        st_y(centroid_geom) as centroid_lat,
        ST_Extent(geom) as bounding_box_bak,
        ST_AsGeoJSON(ST_Envelope(geom)) as bounding_box

    FROM states
    WHERE NAME = 'Ohio'
"""

bbox_data = con.execute(bbox_query).fetchdf()
display(bbox_data)

In [ ]:
# Create map with shape, centroid, and bounding box
bbox_map = folium.Map(location=[bbox_data['centroid_lat'][0], bbox_data['centroid_lon'][0]], zoom_start=7)

# Add state boundary (filled)
folium.GeoJson(bbox_data['geom_outline'][0], style_function=lambda x: {'fillColor': 'lightblue'}).add_to(bbox_map)

# Add bounding box for the state
folium.GeoJson(bbox_data['bounding_box'][0], style_function=lambda x: {'color': 'black', 'fill': False}).add_to(bbox_map)

# Add centroid marker
folium.Marker(location=[bbox_data['centroid_lat'][0], bbox_data['centroid_lon'][0]], icon=folium.Icon(color='blue')).add_to(bbox_map)

save_path = 'maps/bbox_map.html'
bbox_map.save(save_path)
# display(bbox_map)

### Extracting Latitudes and Longitudes
- Many shapes come prepackaged with their centroid or internal point documented as a point geometry or a coordiante pair.
- In the event they do not, you can calculate the centroid of 1 or more geometries.
- You can then extract the coordinates for that centroid
- You may need to do this for certain mapping operations, or to use certain APIs (e.g., reverse geocoding)
- Enables easy filtering operations
    - Show customers east and south of the facility (latitude < facility_lat AND longitude > facility_lon)
    - Warning: Coordinate systems and +/- with lat/lon are just as challenging as working with timezones. Overall it is just a few concepts, but somehow they are really easy to mix up.

In [ ]:
# Extract lat/lon
lat_long_query = """
    SELECT 
        NAME,
        ST_AsGeoJSON(geom) as geom_outline,
        ST_Centroid(geom) as centroid_geom,
        st_x(centroid_geom) as centroid_lon,
        st_y(centroid_geom) as centroid_lat,
        ST_Extent(geom) as bounding_box_bak,
        ST_AsGeoJSON(ST_Envelope(geom)) as bounding_box
    FROM states
    WHERE NAME = 'Ohio'
"""

lat_long_data = con.execute(lat_long_query).fetchdf()
lat_long_data.head()
# display(lat_long_data['NAME', 'centroid_lon', 'centroid_lat', ])

In [ ]:
# Create map with shape, centroid, and bounding box
latlon_map = folium.Map(location=[bbox_data['centroid_lat'][0], bbox_data['centroid_lon'][0]], zoom_start=7)

# Add state boundary (filled)
folium.GeoJson(bbox_data['geom_outline'][0], style_function=lambda x: {'fillColor': 'lightblue'}).add_to(latlon_map)

# Add centroid marker
folium.Marker(location=[bbox_data['centroid_lat'][0], bbox_data['centroid_lon'][0]], icon=folium.Icon(color='blue')).add_to(latlon_map)

folium.PolyLine(        
      locations=[[bbox_data['centroid_lat'][0], -180], [bbox_data['centroid_lat'][0], 180]],
      color='red',
      weight=2,
      dash_array='5, 5'  # optional: makes it dashed
  ).add_to(latlon_map)

folium.PolyLine(
    locations=[[-90, bbox_data['centroid_lon'][0]], [90, bbox_data['centroid_lon'][0]]],                                                                                                                  
    color='green',                                                                                                                                      
    weight=2,
    dash_array='5, 5'
).add_to(latlon_map)

save_path = 'maps/latlon_map.html'
latlon_map.save(save_path)
# display(latlon_map) 


## Practical Applications

### Showing Things on Maps - Shapes & Points

In [ ]:
state_query = """
    SELECT 
        NAME,
        ST_AsGeoJSON(geom) as geom_outline,
        ST_Centroid(geom) as centroid_geom,
        st_x(centroid_geom) as centroid_lon,
        st_y(centroid_geom) as centroid_lat,
        ST_Extent(geom) as bounding_box_bak,
        ST_AsGeoJSON(ST_Envelope(geom)) as bounding_box
    FROM states
"""

state_data = con.execute(state_query).fetchdf()
state_data.head()

In [ ]:
address_point_query = """
SELECT
latitude_best,
longitude_best,
ST_Point (longitude_best, latitude_best) as address_point,
ST_AsGeoJSON(address_point) as geom_point
from dim_addresses
where latitude_best IS NOT NULL AND longitude_best IS NOT NULL
limit 500
"""

address_point_data = con.execute(address_point_query).fetchdf()
address_point_data

#### Map

In [ ]:
# Create map with shape, centroid, and bounding box
states_map = folium.Map(location=[38.28411870779948, -96.05302425184635], zoom_start=5)

# Add Each State
features = []
for idx, row in state_data.iterrows():
    if row['geom_outline']:  # skip nulls
        features.append({
            "type": "Feature",
            "geometry": json.loads(row['geom_outline']),
            "properties": {"name": row.get('NAME', '')}
            })
        
feature_collection = {
    "type": "FeatureCollection",
    "features": features
}

folium.GeoJson(feature_collection).add_to(states_map)

# Add Each Point
features = []
for idx, row in address_point_data.iterrows():
    if row['geom_point']:  # skip nulls
        features.append({
            "type": "Feature",
            "geometry": json.loads(row['geom_point']),
            "properties": "NA"
            # "properties": {"name": row.get('NAME', '')}
            })
        
point_collection = {
    "type": "FeatureCollection",
    "features": features
}

folium.GeoJson(point_collection).add_to(states_map)

save_path = 'maps/states_map.html'
states_map.save(save_path)
# display(states_map) 

### Showing Things on Maps - From Spatial Join

In [ ]:
spatial_join_query = """
SELECT
    *
    from dim_geos
where state_name IN ('Ohio', 'Texas', 'Florida', 'California', 'New York')
"""

spatial_join_data = con.execute(spatial_join_query).fetchdf()
spatial_join_data

#### Map

In [ ]:
# Create map with shape, centroid, and bounding box
spatial_join_map = folium.Map(location=[38.28411870779948, -96.05302425184635], zoom_start=5)

# Add Each State
features = []
for idx, row in state_data.iterrows():
    if row['geom_outline']:  # skip nulls
        features.append({
            "type": "Feature",
            "geometry": json.loads(row['geom_outline']),
            "properties": {"name": row.get('NAME', '')}
            })
        
feature_collection = {
    "type": "FeatureCollection",
    "features": features
}

folium.GeoJson(feature_collection).add_to(spatial_join_map)

# Add Each Point
features = []
for idx, row in spatial_join_data.iterrows():
    if row['geom_point']:  # skip nulls
        features.append({
            "type": "Feature",
            "geometry": json.loads(row['geom_point']),
            "properties": "NA"
            # "properties": {"name": row.get('NAME', '')}
            })
        
point_collection = {
    "type": "FeatureCollection",
    "features": features
}

folium.GeoJson(point_collection).add_to(spatial_join_map)

save_path = 'maps/spatial_join_map.html'
spatial_join_map.save(save_path)

#### Frequency Distribution

Same type of idea, but showing a tabular result - no map required.

In [ ]:
freq_def_query = """
SELECT
    *
    from dim_geos
"""

freq_def_data = con.execute(freq_def_query).fetchdf()
freq_def_data
df = freq_def_data.copy().groupby('state_name').size().reset_index(name='count')
df.sort_values("count", ascending=False)

## Distance Calculations

### Points

Given two points, how far apart are they?

In [ ]:
point_distance_query = """
with

point1 as (
    SELECT
        ST_Point(longitude_best, latitude_best) as point_geom,
    from dim_geos
    WHERE
        (latitude_best = '38.019822' AND longitude_best = '-122.072767')
    ),

point2 as (
    SELECT
        ST_Point(longitude_best, latitude_best) as point_geom,
    from dim_geos
    WHERE
        (latitude_best = '40.831891' AND longitude_best = '-81.889705')
    )

    select
    *,
    ST_Distance_Sphere(point1.point_geom, point2.point_geom) as distance_meters,
    distance_meters/1000 as kilometers,
    kilometers * 0.62137273664981 as miles
    from point1 cross join point2
"""

point_distance_data = con.execute(point_distance_query).fetchdf()
point_distance_data

### How far apart are two shapes?


In [ ]:

centroid_distance_query = """
with

shape1 as (
    SELECT 
        name,
        geom_point
    from stg_shapes__states
    WHERE name = 'Ohio'
),

shape2 as (
    SELECT 
        name,
        geom_point
    from stg_shapes__states
    WHERE name = 'California'
)

select
    shape1.name as shape1_name,
    shape2.name as shape2_name,
    shape1.geom_point as geom_point1,
    shape2.geom_point as geom_point2,
    ST_Distance_Sphere(shape1.geom_point, shape2.geom_point) as distance_meters,
    distance_meters/1000 as kilometers,
    kilometers * 0.62137273664981 as miles
from shape1 cross join shape2
"""

shape_edge_distance_data = con.execute(centroid_distance_query).fetchdf()
shape_edge_distance_data

### Filter Based on Distance

Which states have their centroid within 1000 miles of Ohio?

In [ ]:
# Filter states within 1000 miles of Ohio's centroid
distance_filter_query = """
WITH ohio AS (
    SELECT 
        name,
        geom_point as centroid
    FROM stg_shapes__states
    WHERE name = 'Ohio'
)

SELECT 
    s.name,
    s.state_abbreviation,
    ST_Distance_Sphere(s.geom_point, ohio.centroid) as distance_meters,
    distance_meters/1000 as kilometers,
    kilometers * 0.62137273664981 as miles
FROM stg_shapes__states s
CROSS JOIN ohio
WHERE 
miles <= 1000
  AND
    s.name != 'Ohio'
ORDER BY miles desc
"""

distance_filter_data = con.execute(distance_filter_query).fetchdf()
print(f"States within 1000 miles of Ohio: {len(distance_filter_data)}")
distance_filter_data

In [ ]:
# Visualize states within 500 miles
distance_map = folium.Map(location=[40.0, -85.0], zoom_start=4)

# Get Ohio's geometry for reference
ohio_geom = con.execute("""
    SELECT ST_AsGeoJSON(geom) as geom_outline, 
           st_y(geom_point) as lat, st_x(geom_point) as lon
    FROM stg_shapes__states WHERE name = 'Ohio'
""").fetchdf()

# Get all states within distance
nearby_states = con.execute("""
    WITH ohio AS (
        SELECT geom_point as centroid FROM stg_shapes__states WHERE name = 'Ohio'
    ),
    distances AS (
        SELECT 
            s.name,
            s.state_abbreviation,
            ST_Distance_Sphere(s.geom_point, ohio.centroid) as distance_meters,
            ST_AsGeoJSON(s.geom) as geom_outline
        FROM stg_shapes__states s, ohio
    )
    SELECT 
        name,
        distance_meters,
        distance_meters / 1609.34 as distance_miles,
        geom_outline
    FROM distances
    WHERE distance_meters / 1609.34 <= 500 
    and state_abbreviation NOT IN ('FL', 'MS', 'AL', 'GA')
""").fetchdf()

# Add nearby states (blue)
for idx, row in nearby_states.iterrows():
    folium.GeoJson(
        row['geom_outline'], 
        style_function=lambda x: {'fillColor': 'lightblue', 'color': 'blue', 'weight': 1}
    ).add_to(distance_map)

# Highlight Ohio (red)
folium.GeoJson(
    ohio_geom['geom_outline'][0], 
    style_function=lambda x: {'fillColor': 'red', 'color': 'red', 'weight': 2}
).add_to(distance_map)

# Add Ohio centroid marker
folium.Marker([ohio_geom['lat'][0], ohio_geom['lon'][0]], 
              popup='Ohio Centroid', icon=folium.Icon(color='red')).add_to(distance_map)

# Add 500 mile radius circle (approximate)
folium.Circle(
    location=[ohio_geom['lat'][0], ohio_geom['lon'][0]],
    radius=804672,  # 500 miles in meters
    color='red', fill=False, weight=2, dash_array='10'
).add_to(distance_map)

distance_map.save('maps/distance_filter_map.html')
# distance_map

### Filter Based on Bounding Box

Which state centroids fall within Ohio's bounding box?

Bounding boxes are useful for fast pre-filtering before more expensive spatial operations. This is especially valuable when working with large datasets like zip codes or addresses.

In [ ]:
# Filter state centroids within Ohio's bounding box
bbox_filter_query = """
WITH ohio_bbox AS (
    SELECT ST_Envelope(geom) as bbox
    FROM stg_shapes__states
    WHERE name = 'Ohio'
)
SELECT 
    s.name,
    s.state_abbreviation,
    st_x(s.geom_point) as centroid_lon,
    st_y(s.geom_point) as centroid_lat
FROM stg_shapes__states s, ohio_bbox
WHERE ST_Within(s.geom_point, ohio_bbox.bbox)
ORDER BY s.name
"""

bbox_filter_data = con.execute(bbox_filter_query).fetchdf()
print(f"State centroids within Ohio's bounding box: {len(bbox_filter_data)}")
bbox_filter_data

In [ ]:
# Visualize bounding box filter
bbox_map = folium.Map(location=[40.0, -82.5], zoom_start=6)

# Get Ohio's bounding box
ohio_bbox_geom = con.execute("""
    SELECT ST_AsGeoJSON(ST_Envelope(geom)) as bbox, ST_AsGeoJSON(geom) as outline
    FROM stg_shapes__states WHERE name = 'Ohio'
""").fetchdf()

# Add Ohio outline
folium.GeoJson(ohio_bbox_geom['outline'][0], 
               style_function=lambda x: {'fillColor': 'lightblue', 'color': 'blue'}).add_to(bbox_map)

# Add bounding box
folium.GeoJson(ohio_bbox_geom['bbox'][0], 
               style_function=lambda x: {'color': 'red', 'fill': False, 'weight': 3, 'dashArray': '10'}).add_to(bbox_map)

# Add centroids that fall within
for idx, row in bbox_filter_data.iterrows():
    color = 'green' if row['name'] == 'Ohio' else 'orange'
    folium.CircleMarker(
        location=[row['centroid_lat'], row['centroid_lon']],
        radius=8, color=color, fill=True, popup=row['name']
    ).add_to(bbox_map)

bbox_map.save('maps/bbox_filter_map.html')
# bbox_map

## Spatial Operations - Within and Intersects

These are the core spatial predicates for determining relationships between geometries.


### Method: ST_Within()

`ST_Within(geometry_a, geometry_b)` returns TRUE if geometry_a is **completely contained** within geometry_b. 

- Use case: Find all zip codes entirely within a state boundary
- Stricter than ST_Intersects - the entire shape must be inside

In [ ]:
# ST_Within: Find points completely within Ohio
# Using address points as an example

within_query = """
WITH ohio AS (
    SELECT geom FROM stg_shapes__states WHERE name = 'Ohio'
)
SELECT 
    zip.zcta,
    zip.internal_point_lat,
    zip.internal_point_lon,
    ST_AsGeoJSON(zip.geom_point) as point_geom,
    ST_AsGeoJSON(zip.geom) as geom_outline
FROM stg_shapes__zcta as zip, ohio
WHERE ST_Within(zip.geom_point, ohio.geom)
"""

within_data = con.execute(within_query).fetchdf()
print(f"Points within Ohio: {len(within_data)}")
within_data.head()

In [ ]:
# Visualize ST_Within results
within_map = folium.Map(location=[40.0, -82.5], zoom_start=7)

# Add Ohio boundary
ohio_outline = con.execute("""
    SELECT ST_AsGeoJSON(geom) as outline FROM stg_shapes__states WHERE name = 'Ohio'
""").fetchdf()

folium.GeoJson(ohio_outline['outline'][0], 
               style_function=lambda x: {'color': 'green', 'fillOpacity': 0.3}).add_to(within_map)

# Add zip codes within Ohio as GeoJSON
features = []
for idx, row in within_data.iterrows():
    
    features.append({
            "type": "Feature",
            "geometry": json.loads(row['geom_outline']),
            "properties": {}   
    })

point_collection = {
    "type": "FeatureCollection",
    "features": features
}

folium.GeoJson(point_collection, 
               style_function=lambda x: {'color': 'blue', 'fillColor': 'lightblue', 'fillOpacity': 0.1}).add_to(within_map)


within_map.save('maps/within_map.html')
# within_map

### st_intersects() aaa

In [ ]:
# ST_Intersects: Find all states that share a border with Ohio
intersects_query = """
WITH county AS (
    SELECT geom FROM stg_shapes__counties WHERE name = 'Cuyahoga'
)

SELECT 
    zip.zcta,
    zip.internal_point_lat,
    zip.internal_point_lon,
    ST_AsGeoJSON(zip.geom_point) as point_geom,
    ST_AsGeoJSON(zip.geom) as geom_outline
FROM stg_shapes__zcta as zip, county
WHERE ST_Intersects(zip.geom, county.geom)
ORDER BY zip.zcta
"""

intersects_data = con.execute(intersects_query).fetchdf()
print(f"States that intersect with selected county: {len(intersects_data)}")
intersects_data.head()

In [ ]:
# Visualize ST_Intersects results
intersects_map = folium.Map(location=[40.0, -82.5], zoom_start=8)

# Add County Outline (highlighted)
county_outline = con.execute("""
    SELECT ST_AsGeoJSON(geom) as outline FROM stg_shapes__counties WHERE name = 'Cuyahoga'
""").fetchdf()

folium.GeoJson(county_outline['outline'][0], 
               style_function=lambda x: {'color': 'green', 'fillOpacity': 0.1}).add_to(intersects_map)

# Add intersecting states
for idx, row in intersects_data.iterrows():
    folium.GeoJson(
        row['geom_outline'], 
        style_function=lambda x: {'fillColor': 'lightblue', 'color': 'blue', 'fillOpacity': 0.3},
        tooltip=row['zcta']
    ).add_to(intersects_map)

intersects_map.save('maps/intersects_map.html')
# intersects_map


## Other Spatial Methods

### Creating Lines

Lines connect two or more points. Useful for:
- Route visualization
- Showing connections between locations
- Finding what geometries a path crosses through

In [ ]:
# Create a line from Columbus, OH to Cleveland, OH and find what it crosses
line_query = """
WITH route AS (
    SELECT ST_MakeLine(
        ST_Point(-82.9988, 39.9612),  -- Columbus
        ST_Point(-81.6944, 41.4993)   -- Cleveland
    ) as line_geom
)
SELECT 
    s.name,
    s.state_abbreviation,
    ST_AsGeoJSON(route.line_geom) as line_geojson
FROM stg_shapes__states s, route
WHERE ST_Intersects(s.geom, route.line_geom)
"""

line_data = con.execute(line_query).fetchdf()
print("States crossed by Columbus-Cleveland line:")
line_data.head()

In [ ]:
# Cross-country line: Los Angeles to New York
cross_country_query = """
WITH route AS (
    SELECT ST_MakeLine(
        ST_Point(-118.2437, 34.0522),  -- Los Angeles
        ST_Point(-74.0060, 40.7128)    -- New York
    ) as line_geom
)
SELECT 
    s.name,
    s.state_abbreviation,
    ST_AsGeoJSON(s.geom) as state_geom,
    ST_AsGeoJSON(route.line_geom) as line_geojson
FROM stg_shapes__states s, route
WHERE ST_Intersects(s.geom, route.line_geom)
  AND s.region <> '9'  -- Exclude territories
ORDER BY st_x(s.geom_point)  -- Order west to east
"""

cross_country_data = con.execute(cross_country_query).fetchdf()
print(f"States crossed LA to NYC: {len(cross_country_data)}")
cross_country_data[['name', 'state_abbreviation']]

In [ ]:
# Visualize the cross-country route
line_map = folium.Map(location=[38.0, -98.0], zoom_start=4)

# Add crossed states
for idx, row in cross_country_data.iterrows():
    folium.GeoJson(
        row['state_geom'], 
        style_function=lambda x: {'fillColor': 'yellow', 'color': 'orange', 'fillOpacity': 0.5},
        tooltip=row['name']
    ).add_to(line_map)

# Add the line
folium.GeoJson(
    cross_country_data['line_geojson'][0], 
    style_function=lambda x: {'color': 'red', 'weight': 3}
).add_to(line_map)

# Add start/end markers
folium.Marker([34.0522, -118.2437], popup='Los Angeles', icon=folium.Icon(color='green')).add_to(line_map)
folium.Marker([40.7128, -74.0060], popup='New York', icon=folium.Icon(color='red')).add_to(line_map)

line_map.save('maps/line_route_map.html')
line_map

### Combining Shapes (ST_Union)

`ST_Union` merges multiple geometries into a single geometry. Useful for:
- Creating custom regions from existing boundaries
- Merging adjacent zip codes or counties
- Building service territories

In [ ]:
# Combine Great Lakes states into one region
union_query = """
SELECT 
    'Great Lakes Region' as region_name,
    ST_Union_Agg(geom) as combined_geom,
    ST_AsGeoJSON(ST_Union_Agg(geom)) as combined_geojson,
    STRING_AGG(state_abbreviation, ', ') as states_included
FROM stg_shapes__states
WHERE name IN ('Ohio', 'Michigan', 'Indiana', 'Illinois', 'Wisconsin', 'Minnesota')
"""

union_data = con.execute(union_query).fetchdf()
print(f"Combined region includes: {union_data['states_included'][0]}")
union_data[['region_name', 'states_included']]

In [ ]:
# Visualize the combined Great Lakes region
union_map = folium.Map(location=[42.0, -87.0], zoom_start=5)

# Add the combined region as a single shape
folium.GeoJson(
    union_data['combined_geojson'][0], 
    style_function=lambda x: {'fillColor': 'blue', 'color': 'darkblue', 'fillOpacity': 0.4, 'weight': 2},
    tooltip='Great Lakes Region'
).add_to(union_map)

union_map.save('maps/union_map.html')
union_map

### Not Just States...

The same spatial join pattern works with any shapefile:
- **Census Tracts** - demographic analysis
- **Congressional Districts** - political geography  
- **School Districts** - education planning
- **Zip Codes (ZCTA)** - market analysis
- **Custom regions** - business territories

## H3 Geospatial Index

### What is H3?

H3 is a hierarchical hexagonal grid system developed by Uber. It divides the world into hexagonal cells at multiple resolutions:
- **Resolution 0**: ~4,250,000 kmÂ² per cell (continental scale)
- **Resolution 4**: ~1,770 kmÂ² per cell (~1 per state)
- **Resolution 9**: ~105 mÂ² per cell (building scale)
- **Resolution 15**: ~0.9 mÂ² per cell (sub-meter precision)

Benefits:
- Uniform cell shapes (unlike lat/lon grids)
- Hierarchical - each cell contains ~7 child cells
- Fast aggregation and neighbor lookups
- No edge effects at poles

### H3 Resolution Examples

In [ ]:
# H3 at different resolutions for Columbus, OH
columbus_lat, columbus_lon = 39.9612, -82.9988

h3_resolution_query = f"""
SELECT 
    res,
    h3_latlng_to_cell({columbus_lat}, {columbus_lon}, res) as h3_index,
    h3_cell_to_boundary_wkt(h3_index) as boundary_wkt
FROM (VALUES (1), (3), (5), (7), (9)) AS t(res)
"""

h3_resolutions = con.execute(h3_resolution_query).fetchdf()
h3_resolutions

### Point Data - Convert to H3 Index

In [ ]:
# Convert address points to H3 indices
h3_points_query = """
SELECT 
    latitude_best,
    longitude_best,
    h3_latlng_to_cell(latitude_best, longitude_best, 5) as h3_res5,
    h3_latlng_to_cell(latitude_best, longitude_best, 7) as h3_res7
FROM dim_addresses
WHERE latitude_best IS NOT NULL
LIMIT 20
"""

h3_points = con.execute(h3_points_query).fetchdf()
h3_points

### Point Data - Convert to H3 Index + Boundary

In [ ]:
# Get H3 cell boundaries as geometries
h3_boundary_query = """
SELECT 
    latitude_best,
    longitude_best,
    h3_latlng_to_cell(latitude_best, longitude_best, 5) as h3_index,
    h3_cell_to_boundary_wkt(h3_index) as h3_boundary_wkt,
    ST_GeomFromText(h3_boundary_wkt) as h3_geom,
    ST_AsGeoJSON(h3_geom) as h3_geojson
FROM dim_addresses
WHERE latitude_best IS NOT NULL
LIMIT 50
"""

h3_boundaries = con.execute(h3_boundary_query).fetchdf()
print(f"Unique H3 cells: {h3_boundaries['h3_index'].nunique()}")
h3_boundaries.head()

### Point Data - Aggregate by H3 Cell

H3 makes aggregation simple - just group by the H3 index.

In [ ]:
# Aggregate points by H3 cell
h3_aggregate_query = """
WITH h3_points AS (
    SELECT 
        latitude_best,
        longitude_best,
        h3_latlng_to_cell(latitude_best, longitude_best, 4) as h3_index
    FROM dim_addresses
    WHERE latitude_best IS NOT NULL
)
SELECT 
    h3_index,
    COUNT(*) as point_count,
    h3_cell_to_boundary_wkt(h3_index) as boundary_wkt,
    ST_AsGeoJSON(ST_GeomFromText(boundary_wkt)) as h3_geojson
FROM h3_points
GROUP BY h3_index
HAVING COUNT(*) >= 1
ORDER BY point_count DESC
"""

h3_aggregated = con.execute(h3_aggregate_query).fetchdf()
h3_aggregated

In [ ]:
# Visualize H3 aggregation
h3_map = folium.Map(location=[38.0, -98.0], zoom_start=4)

# Color scale based on point count
max_count = h3_aggregated['point_count'].max()

for idx, row in h3_aggregated.iterrows():
    # Color intensity based on count
    intensity = row['point_count'] / max_count
    color = f'#{int(255*intensity):02x}00{int(255*(1-intensity)):02x}'
    
    folium.GeoJson(
        row['h3_geojson'],
        style_function=lambda x, c=color: {'fillColor': c, 'color': 'black', 'fillOpacity': 0.6, 'weight': 1},
        tooltip=f"Count: {row['point_count']}"
    ).add_to(h3_map)

h3_map.save('maps/h3_aggregate_map.html')
h3_map

Compare to: (Subset of data because of marker limitations)

![H3 Map Markers](h3-map-markers.png)

### H3 Cells Intersecting with Shapes

You can combine H3 with traditional shapefiles to find which H3 cells are within, or intersect geographies of interst - e.g. summarize customer activity by H3 cell, for a specific state.

In [ ]:
# Find H3 cells (res 4) that cover Ohio
h3_ohio_query = """
WITH ohio AS (
    SELECT geom, ST_AsGeoJSON(geom) as geom_json FROM stg_shapes__states WHERE name = 'Ohio'
),
h3_cells AS (
    SELECT 
        h3_latlng_to_cell(latitude_best, longitude_best, 4) as h3_index
    FROM dim_addresses
    WHERE latitude_best IS NOT NULL
    GROUP BY 1
)
SELECT DISTINCT
    h.h3_index,
    h3_cell_to_boundary_wkt(h.h3_index) as boundary_wkt,
    ST_AsGeoJSON(ST_GeomFromText(boundary_wkt)) as h3_geojson
FROM h3_cells h, ohio o
WHERE ST_Intersects(ST_GeomFromText(h3_cell_to_boundary_wkt(h.h3_index)), o.geom)
"""

h3_ohio = con.execute(h3_ohio_query).fetchdf()
print(f"H3 cells (res 4) covering Ohio: {len(h3_ohio)}")
h3_ohio

In [ ]:
# Visualize H3 cells over Ohio
h3_ohio_map = folium.Map(location=[40.0, -82.5], zoom_start=7)

# Add Ohio boundary
folium.GeoJson(
    ohio_outline['outline'][0], 
    style_function=lambda x: {'fillColor': 'lightblue', 'color': 'blue', 'fillOpacity': 0.2, 'weight': 2}
).add_to(h3_ohio_map)

# Add H3 hexagons
for idx, row in h3_ohio.iterrows():
    folium.GeoJson(
        row['h3_geojson'],
        style_function=lambda x: {'fillColor': 'orange', 'color': 'red', 'fillOpacity': 0.4, 'weight': 1}
    ).add_to(h3_ohio_map)

h3_ohio_map.save('maps/h3_ohio_map.html')
h3_ohio_map

## Summary

This notebook demonstrated key geospatial concepts using DuckDB's spatial extension:

1. **Basic Shape Operations**: Centroid, bounding box, coordinate extraction
2. **Distance Filtering**: Finding shapes within a radius
3. **Bounding Box Filtering**: Fast pre-filtering for spatial queries
4. **Spatial Predicates**: ST_Within (completely inside) vs ST_Intersects (any overlap)
5. **Line Geometries**: Creating routes and finding intersections
6. **Shape Combinations**: ST_Union for merging geometries
7. **Spatial Joins**: Enriching point data with shape properties
8. **H3 Indexing**: Hexagonal grid system for aggregation and analysis

**Key Takeaway**: These operations work entirely within SQL - no mapping application required for the analysis, though visualizations help communicate results.